# Associate the extracted places with Cliopatria polities

Builds the benchmark dataset: for each benchmark individual and life stage, one or more Cliopatria polities for the place extracted from the biography in `02_extract_places_from_biographies.ipynb`.

1. The Cliopatria polities whose territory holds each place during the stage years given by the biography are listed as candidates.
2. Claude Code agents choose among the candidates the polity or polities of each stage.

Agent outputs are cached in `data/association/`. The comparison with Cultura is in `04_benchmark_results.ipynb`.

In [1]:
import json
import subprocess
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import polars as pl
from IPython.display import Markdown
from shapely import STRtree, make_valid, points
from shapely.geometry import shape
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent
FOLDER = ROOT / "notebooks" / "notebooks_polity_validation" / "data"
PROMPTS = ROOT / "notebooks" / "notebooks_polity_validation" / "prompts"
EXTRACTION = FOLDER / "extraction"
ASSOCIATION = FOLDER / "association"
LOGS = FOLDER / "logs"
CLIOPATRIA = ROOT / "data" / "cliopatria_data" / "cliopatria_V2" / "cliopatria_polities_only_v3.geojson"
OUT = FOLDER / "benchmark_life_stage_polities.parquet"
MODEL = "claude-opus-5-5"
STAGES = ["pre_active", "active", "post_active"]
ASSOCIATION_GROUP = 40
WORKERS = 10
LIMIT = None
for folder in (ASSOCIATION, LOGS):
    folder.mkdir(exist_ok=True)

/Users/charlesdedampierre/Desktop/Rsearch Folder/cultura/cultura_database/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Benchmark individuals

The individuals whose biography could be downloaded (`scripts/5_texts.py`), one text file each in `data/texts/`.

In [2]:
people = pl.DataFrame([{key: person[key] for key in ("qid", "label", "dictionary_name", "biography_url")}
                       for person in json.loads((FOLDER / "benchmark_texts.json").read_text(encoding="utf-8"))])
if LIMIT:
    people = people.head(LIMIT)
assert people["qid"].is_unique().all()
assert all((FOLDER / "texts" / f"{qid}.txt").exists() for qid in people["qid"])
print(f"{len(people)} individuals")
people.head()

547 individuals


qid,label,dictionary_name,biography_url
str,str,str,str
"""Q9333""","""Laozi""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q4604""","""Confucius""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q47739""","""Zhuang Zhou""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q216072""","""Xunzi""","""Stanford Encyclopedia of Philo…","""https://plato.stanford.edu/ent…"
"""Q7212224""","""Luoxia Hong""","""MacTutor History of Mathematic…","""https://mathshistory.st-andrew…"


### Claude Code runner

Each agent is a headless Claude Code session (`claude -p`) started from the repository root, with only the tools it needs; its full output is kept in `data/logs/`.

In [3]:
def claude(prompt, tools, allowed, log):
    """Run one headless Claude Code session and return its final reply."""
    command = ["claude", "-p", "--model", MODEL, "--output-format", "json", *(["--allowedTools", *allowed] if allowed else []), "--tools", *tools]
    result = subprocess.run(command, input=prompt, cwd=ROOT, capture_output=True, text=True, timeout=3600)
    log.write_text(result.stdout + result.stderr, encoding="utf-8")
    return json.loads(result.stdout)["result"]


def groups(qids, size):
    return [qids[start:start + size] for start in range(0, len(qids), size)]


def done(path, qids):
    return path.exists() and set(qids) <= set(json.loads(path.read_text(encoding="utf-8")))


def run_all(task, jobs, label):
    """Run task(number, qids) for every job in parallel; a failed group is reported and left for the next run."""
    def safe(job):
        try:
            task(*job)
        except Exception as error:
            print(f"{label} group {job[0]:02d} failed: {error}")
    with ThreadPoolExecutor(WORKERS) as pool:
        list(tqdm(pool.map(safe, jobs), total=len(jobs), desc=label))

## Extracted places

The extraction of `02_extract_places_from_biographies.ipynb`.

In [4]:
qids = set(people["qid"])
extraction = {qid: person for path in sorted(EXTRACTION.glob("agent_*.json"))
              for qid, person in json.loads(path.read_text(encoding="utf-8")).items() if qid in qids}
print(f"{len(extraction)} / {len(people)} individuals extracted")
filled = pl.DataFrame([{"stage": stage, "place": bool(person.get(stage)),
                        "years": bool(person.get(stage) and any(year is not None for year in person[stage].get("years") or []))}
                       for person in extraction.values() for stage in STAGES])
filled.group_by("stage").agg(individuals=pl.len(), with_place=pl.col("place").sum(), with_years=pl.col("years").sum()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

547 / 547 individuals extracted


stage,individuals,with_place,with_years
str,u32,u32,u32
"""pre_active""",547,437,372
"""active""",547,528,503
"""post_active""",547,392,366


## Step 1: Cliopatria candidates

For each stage, the Cliopatria polities whose territory contains a point of the extracted place and that existed during the stage years given by the biography. A missing bound is taken from the life years the biography gives for the other stages; a stage with no years at all keeps every polity that ever held the place. Each candidate comes with its years of existence and the polities it belongs to (`MemberOf` and `Components` of Cliopatria).

In [5]:
features = json.loads(CLIOPATRIA.read_text())["features"]
polygons = STRtree(make_valid([shape(feature["geometry"]) for feature in features]))
properties = [feature["properties"] for feature in features]
parents = {}
for polity in properties:
    for parent in filter(None, polity["MemberOf"].split(";")):
        parents.setdefault(polity["Name"], set()).add(parent)
    for child in filter(None, polity["Components"].split(";")):
        parents.setdefault(child, set()).add(polity["Name"])
print(f"{len(features):,} Cliopatria polygons, {len({polity['Name'] for polity in properties}):,} polities")

13,755 Cliopatria polygons, 1,633 polities


In [6]:
def stage_years(person, stage):
    known = [year for name in STAGES if person.get(name) for year in person[name].get("years") or [] if year is not None]
    start, end = (list(person[stage].get("years") or []) + [None, None])[:2]
    if known:
        start = min(known) if start is None else start
        end = max(known) if end is None else end
    return start, end


def candidates(person, stage):
    """Cliopatria polities holding a point of the stage place during the stage years."""
    spots = person[stage].get("points") or []
    if not spots:
        return []
    start, end = stage_years(person, stage)
    found = {}
    for spot, index in zip(*polygons.query(points([(longitude, latitude) for _, longitude, latitude in spots]), predicate="within")):
        polity = properties[index]
        if start is not None and (polity["ToYear"] < start or polity["FromYear"] > end):
            continue
        entry = found.setdefault(polity["Name"], {"name": polity["Name"], "from": polity["FromYear"], "to": polity["ToYear"],
                                                  "member_of": sorted(parents.get(polity["Name"], ())), "points": []})
        entry["from"], entry["to"] = min(entry["from"], polity["FromYear"]), max(entry["to"], polity["ToYear"])
        if spots[spot][0] not in entry["points"]:
            entry["points"].append(spots[spot][0])
    return sorted(found.values(), key=lambda entry: entry["from"])


association_input = {}
for qid, person in extraction.items():
    for stage in STAGES:
        if person.get(stage):
            found = candidates(person, stage)
            if found:
                association_input.setdefault(qid, {})[stage] = {
                    "place": person[stage]["place"], "written": person[stage]["written"], "written_en": person[stage]["written_en"],
                    "years": person[stage].get("years"), "extracts": person[stage]["verbatim_en"], "candidates": found}
counts = pl.DataFrame([{"stage": stage, "candidates": len(entry["candidates"])} for stages in association_input.values() for stage, entry in stages.items()])
print(f"{len(association_input)} individuals with at least one stage to associate")
counts.group_by("stage").agg(stages=pl.len(), median_candidates=pl.col("candidates").median(), max_candidates=pl.col("candidates").max()).sort(pl.col("stage").cast(pl.Enum(STAGES)))

523 individuals with at least one stage to associate


stage,stages,median_candidates,max_candidates
str,u32,f64,i64
"""pre_active""",397,2.0,61
"""active""",496,2.0,66
"""post_active""",346,1.0,48


### Place and years used for the candidates

The candidates come only from the biography, never from Cultura:

- **place**: the coordinates the extraction agent gives for each place of the stage (several places allowed); a polity is kept when its territory contains one of the points;
- **years**: each stage is filtered on its own years (pre-active years for the pre-active stage, active years for the active stage, post-active years for the post-active stage), as the biography gives them. A missing bound is filled with the earliest or latest year the biography gives for the other stages; a stage whose biography gives no year at all keeps every polity that ever held the place, and Claude then chooses from the extracts and its historical knowledge.

The table counts, for the stages with a place, which year window was used.

In [7]:
windows = pl.DataFrame([{"stage": stage, "years": person[stage].get("years") or [None, None]}
                        for person in extraction.values() for stage in STAGES if person.get(stage)])
windows = windows.with_columns(window=pl.when(pl.col("years").list.drop_nulls().list.len() == 2).then(pl.lit("both stage years"))
                               .when(pl.col("years").list.drop_nulls().list.len() == 1).then(pl.lit("one stage year, other filled from the life"))
                               .otherwise(pl.lit("no year: all years")))
(windows.group_by("stage", "window").len().pivot(on="window", index="stage", values="len").fill_null(0)
 .sort(pl.col("stage").cast(pl.Enum(STAGES)))
 .select("stage", "both stage years", "one stage year, other filled from the life", "no year: all years"))

stage,both stage years,"one stage year, other filled from the life",no year: all years
str,u32,u32,u32
"""pre_active""",240,132,65
"""active""",400,103,25
"""post_active""",324,42,26


## Step 2: polity association

### Association prompt

In [8]:
ASSOCIATION_PROMPT = (PROMPTS / "biography_cliopatria_association.md").read_text(encoding="utf-8").split("---", 1)[1]
Markdown(ASSOCIATION_PROMPT)



You are associating places extracted from historians' biographies with historical polities of the Cliopatria dataset, for a validation benchmark (Nature Scientific Data paper). Do not open any file and do not use the web: everything you need is below.

Each individual's life is split into three stages: pre_active (birth to the start of the active life), active (the active life: works, offices, professional activity) and post_active (end of the active life to death). For each individual and stage you are given the place extracted from the biography (as written in the text and in English), the years of the stage according to the biography (null when unknown), the supporting extracts, and the candidate Cliopatria polities: each with its name, its years of existence in Cliopatria, the polities it belongs to ("member_of") and the extracted point(s) its territory contains.

For each stage, choose the polity or polities that governed the place during the stage years:
- choose only among the candidates of that stage, and copy their names exactly;
- one polity per distinct place of the stage (a stage with several places may get several polities);
- for a place, prefer the smallest polity, the most specific one in the member_of hierarchy, that held the place during most of the stage years;
- use the extracts and your historical knowledge to decide which candidate fits the place at the time (for example, a city of a contested border region);
- if no candidate fits, give an empty list.

Candidates (JSON, QID -> stage -> data):

{CANDIDATES}

Reply with only one JSON object, no text around it: QID -> {"pre_active": {...}, "active": {...}, "post_active": {...}}, every QID and stage of the input, each stage object having:
- "polities": list of chosen candidate names, most important first
- "confidence": "high" | "medium" | "low"
- "reasoning": one short sentence


### Run the association agents

Groups of `ASSOCIATION_GROUP` individuals; the candidates are written in the prompt, so the agents have no tools. An individual left out of an agent's reply is sent again in the next run and merged into the group's file. A polity name that is not one of the stage candidates is dropped.

In [9]:
def associate(number, qids):
    path = ASSOCIATION / f"agent_{number:02d}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    todo = [qid for qid in qids if qid not in answer]
    if todo:
        prompt = ASSOCIATION_PROMPT.replace("{CANDIDATES}", json.dumps({qid: association_input[qid] for qid in todo}, ensure_ascii=False, indent=1))
        reply = claude(prompt, [""], [], LOGS / f"association_{number:02d}_{len(answer)}.log")
        answer |= json.loads(reply[reply.index("{"):reply.rindex("}") + 1])
        path.write_text(json.dumps(answer, ensure_ascii=False, indent=2), encoding="utf-8")


association_jobs = list(enumerate(groups(sorted(association_input), ASSOCIATION_GROUP)))
for attempt in range(2):
    run_all(associate, association_jobs, "association")
missing = [number for number, qids in association_jobs if not done(ASSOCIATION / f"agent_{number:02d}.json", qids)]
print(f"{len(association_jobs) - len(missing)} / {len(association_jobs)} groups associated; missing: {missing}")

association: 100%|██████████| 14/14 [00:00<00:00, 175284.35it/s]

14 / 14 groups associated; missing: []


In [10]:
association = {qid: stages for path in sorted(ASSOCIATION.glob("agent_*.json"))
               for qid, stages in json.loads(path.read_text(encoding="utf-8")).items() if qid in association_input}
dropped = []
for qid, stages in association.items():
    for stage, answer in stages.items():
        allowed = {entry["name"] for entry in association_input[qid].get(stage, {}).get("candidates", [])}
        dropped += [(qid, stage, name) for name in answer.get("polities", []) if name not in allowed]
        answer["polities"] = [name for name in answer.get("polities", []) if name in allowed]
print(f"{len(association)} / {len(association_input)} individuals associated; {len(dropped)} polity names not among the candidates dropped")
dropped[:10]

523 / 523 individuals associated; 0 polity names not among the candidates dropped


[]

### Smallest linked polity

A polity chosen by Claude is replaced by its smallest linked candidate: a candidate of the stage that belongs to it in the Cliopatria hierarchy (`MemberOf`, `Components`) and holds the same point, existing for the most stage years. Claude's own choice is kept in `claude_polities`.

In [11]:
def ancestors(name):
    found, todo = set(), [name]
    while todo:
        for parent in parents.get(todo.pop(), ()):
            if parent not in found:
                found.add(parent)
                todo.append(parent)
    return found


def smallest(qid, stage, name):
    """Smallest candidate of the stage linked to name in the hierarchy and holding one of its points."""
    candidates = {entry["name"]: entry for entry in association_input[qid][stage]["candidates"]}
    children = [other for other in candidates if name in ancestors(other) and set(candidates[other]["points"]) & set(candidates[name]["points"])]
    leaves = [child for child in children if not any(child in ancestors(other) for other in children)]
    if not leaves:
        return name
    start, end = stage_years(extraction[qid], stage)
    overlap = lambda child: 0 if start is None else min(candidates[child]["to"], end) - max(candidates[child]["from"], start)
    return max(leaves, key=overlap)


replaced = []
for qid, stages in association.items():
    for stage, answer in stages.items():
        answer["claude_polities"] = list(answer["polities"])
        answer["polities"] = list(dict.fromkeys(smallest(qid, stage, name) for name in answer["polities"]))
        replaced += [(qid, stage, before, after) for before, after in zip(answer["claude_polities"], map(lambda name: smallest(qid, stage, name), answer["claude_polities"])) if before != after]
print(f"{len(replaced)} polities replaced by a smaller linked polity")
pl.DataFrame(replaced, schema=["qid", "stage", "claude_polity", "smallest_polity"], orient="row").head(10)

10 polities replaced by a smaller linked polity


qid,stage,claude_polity,smallest_polity
str,str,str,str
"""Q150320""","""post_active""","""(Holy Roman Empire)""","""Kingdom of Bohemia"""
"""Q161958""","""post_active""","""(Holy Roman Empire)""","""Teutonic Order"""
"""Q169288""","""pre_active""","""(Kingdom of France)""","""Duchy of Aquitaine"""
"""Q312372""","""active""","""(Kingdom of France)""","""County of Vermandois"""
"""Q334806""","""pre_active""","""(Allegiance of Ostrogothic Kin…","""Eastern Roman Empire"""
"""Q366198""","""active""","""(Anglo-Saxon England)""","""Kingdom of Kent"""
"""Q560531""","""pre_active""","""(Anglo-Saxon England)""","""Kingdom of Mercia"""
"""Q91661""","""pre_active""","""(Personal union of Spanish Emp…","""Habsburg Monarchy"""
"""Q919247""","""post_active""","""(Carolingian Empire)""","""West Franks"""


## Final benchmark dataset

One row per individual and stage, with the place and years from the biography and one or more Cliopatria polities (brackets of Cliopatria names removed, as in Cultura). Saved to `data/benchmark_life_stage_polities.parquet`.

In [12]:
rows = []
for qid, person in extraction.items():
    for stage in STAGES:
        extracted = person.get(stage) or {}
        answer = association.get(qid, {}).get(stage) or {}
        start, end = (extracted.get("years") or [None, None]) if extracted else (None, None)
        rows.append({
            "qid": qid, "stage": stage, "biography_start_year": start, "biography_end_year": end,
            "place": extracted.get("place"), "place_as_written": extracted.get("written"), "place_as_written_english": extracted.get("written_en"),
            "verbatim": extracted.get("verbatim", []), "verbatim_english": extracted.get("verbatim_en", []),
            "coordinates": [f"{name} ({latitude}, {longitude})" for name, longitude, latitude in extracted.get("points") or []],
            "extraction_confidence": extracted.get("confidence"), "extraction_reasoning": extracted.get("reasoning"),
            "candidate_polities": list(dict.fromkeys(entry["name"].strip("()") for entry in association_input.get(qid, {}).get(stage, {}).get("candidates", []))),
            "claude_polities": list(dict.fromkeys(name.strip("()") for name in answer.get("claude_polities", []))),
            "polities": list(dict.fromkeys(name.strip("()") for name in answer.get("polities", []))),
            "association_confidence": answer.get("confidence"), "association_reasoning": answer.get("reasoning"), "model_name": MODEL,
        })
benchmark = (pl.DataFrame(rows, schema_overrides={"biography_start_year": pl.Int64, "biography_end_year": pl.Int64})
             .join(people, on="qid", how="left").sort("qid", pl.col("stage").cast(pl.Enum(STAGES))))
assert benchmark.select("qid", "stage").is_unique().all()
assert len(benchmark) == 3 * len(extraction)
benchmark.write_parquet(OUT)
print(f"{len(benchmark)} rows ({benchmark['qid'].n_unique()} individuals) saved to {OUT.relative_to(ROOT)}")
benchmark.filter(pl.col("polities").list.len() > 0).sample(5, seed=42).select("label", "stage", "biography_start_year", "biography_end_year", "place", "polities", "association_confidence")

1641 rows (547 individuals) saved to notebooks/notebooks_polity_validation/data/benchmark_life_stage_polities.parquet


label,stage,biography_start_year,biography_end_year,place,polities,association_confidence
str,str,i64,i64,str,list[str],str
"""Isaac de Beausobre""","""pre_active""",1659,1683,"""Niort; Saumur""","[""Kingdom of France""]","""high"""
"""Adalbert of Prague""","""active""",982,997,"""Prague; Rome; Monte Cassino; B…","[""Duchy of Bohemia"", ""Papal States"", … ""Kingdom of Poland""]","""medium"""
"""Georg Carstensen""","""active""",1835,1857,"""Paris; Philadelphia; Copenhage…","[""United States of America"", ""Bourbon Kingdom of France""]","""medium"""
"""Duris of Samos""","""pre_active""",-340,null,"""Samos""","[""Second Athenian League""]","""low"""
"""Antonín Vranický""","""pre_active""",1761,1782,"""Nová Říše; Brno; Vienna""","[""Habsburg Monarchy""]","""high"""


### Polities per stage

Stages with no polity (no place in the biography, or no Cliopatria polity for it), one polity, or several.

In [13]:
(benchmark.with_columns(n=pl.col("polities").list.len())
 .group_by("stage").agg(no_polity=(pl.col("n") == 0).sum(), one_polity=(pl.col("n") == 1).sum(), several_polities=(pl.col("n") > 1).sum())
 .sort(pl.col("stage").cast(pl.Enum(STAGES))))

stage,no_polity,one_polity,several_polities
str,u32,u32,u32
"""pre_active""",163,301,83
"""active""",59,291,197
"""post_active""",214,329,4


### Structure check

Column types, three stages per individual, chosen polities among the candidates, and no polity without a place.

In [14]:
saved = pl.read_parquet(OUT)
expected = {"qid": pl.String, "stage": pl.String, "biography_start_year": pl.Int64, "biography_end_year": pl.Int64, "place": pl.String,
            "verbatim": pl.List(pl.String), "candidate_polities": pl.List(pl.String), "claude_polities": pl.List(pl.String), "polities": pl.List(pl.String),
            "association_confidence": pl.String, "label": pl.String, "dictionary_name": pl.String, "biography_url": pl.String}
assert all(saved.schema[column] == dtype for column, dtype in expected.items()), {column: saved.schema[column] for column in expected}
assert saved.group_by("qid").agg(pl.col("stage").sort()).filter(pl.col("stage") != sorted(STAGES)).is_empty()
assert saved.filter(pl.col("polities").list.set_difference("candidate_polities").list.len() > 0).is_empty()
assert all(saved.filter(pl.col(column).list.n_unique() < pl.col(column).list.len()).is_empty() for column in ("candidate_polities", "claude_polities", "polities"))
assert saved.filter(pl.col("place").is_null() & (pl.col("polities").list.len() > 0)).is_empty()
assert saved.filter(pl.col("biography_start_year") > pl.col("biography_end_year")).height == 0, "start after end"
print(f"structure ok: {saved.height} rows, {saved['qid'].n_unique()} individuals, {saved.width} columns")
saved.schema

structure ok: 1641 rows, 547 individuals, 21 columns


Schema([('qid', String),
        ('stage', String),
        ('biography_start_year', Int64),
        ('biography_end_year', Int64),
        ('place', String),
        ('place_as_written', String),
        ('place_as_written_english', String),
        ('verbatim', List(String)),
        ('verbatim_english', List(String)),
        ('coordinates', List(String)),
        ('extraction_confidence', String),
        ('extraction_reasoning', String),
        ('candidate_polities', List(String)),
        ('claude_polities', List(String)),
        ('polities', List(String)),
        ('association_confidence', String),
        ('association_reasoning', String),
        ('model_name', String),
        ('label', String),
        ('dictionary_name', String),
        ('biography_url', String)])